# SmartCampus AI — Colab LLM Inference API

This notebook serves the already fine-tuned Qwen3 LoRA adapter as an independent FastAPI service. The existing SmartCampus backend calls this API; the browser should continue to call the existing backend. No training, adapter writes/merges, retrieval, embeddings, or database are used here.

Run cells from top to bottom. A Colab T4 GPU and the Drive adapter are required. The ngrok URL is temporary and changes when a tunnel is restarted.

In [ ]:
%pip -q install "transformers>=4.51.3,<5" "peft>=0.15,<1" "accelerate>=1.6" "bitsandbytes>=0.45" sentencepiece fastapi "uvicorn[standard]" pyngrok requests

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

In [ ]:
BASE_MODEL_NAME = "ESPRIT-Group/ESPRIT-Derja-Qwen3-8B-v2"
ADAPTER_DIR = "/content/drive/MyDrive/SmartCampusAI/models/derja-qwen3-8b-smartcampus-qlora/best_adapter"
SERVER_HOST = "0.0.0.0"
SERVER_PORT = 8000
CHAT_ENDPOINT = "/chat"
HEALTH_ENDPOINT = "/health"
API_VERSION = "1.0.0"
MAX_NEW_TOKENS = 400
TEMPERATURE = 0.3
TOP_P = 0.9
REPETITION_PENALTY = 1.08
MAX_HISTORY_TURNS = 8
MAX_CONTEXT_TOKENS = 32768

In [ ]:
from pathlib import Path
adapter_path = Path(ADAPTER_DIR)
if not adapter_path.is_dir():
    raise FileNotFoundError(f"LoRA adapter directory not found: {ADAPTER_DIR}. Check that Drive is mounted and the path points to best_adapter.")
adapter_files = [p.name for p in adapter_path.iterdir() if p.is_file()]
if "adapter_config.json" not in adapter_files or not any(name.startswith("adapter_model.") for name in adapter_files):
    raise FileNotFoundError("Adapter files are incomplete: expected adapter_config.json and adapter_model.safetensors or adapter_model.bin.")
print(f"Adapter found ({len(adapter_files)} files); saved checkpoint will be read-only.")

In [ ]:
import torch
if not torch.cuda.is_available():
    raise RuntimeError("CUDA GPU unavailable. In Colab choose Runtime > Change runtime type > T4 GPU, then rerun from the start.")
GPU_NAME = torch.cuda.get_device_name(0)
print(f"GPU: {GPU_NAME}; VRAM: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GiB; CUDA: {torch.version.cuda}")

In [ ]:
from transformers import BitsAndBytesConfig
quantization_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

In [ ]:
from google.colab import userdata
from transformers import AutoTokenizer

# Optional: private/gated Hugging Face repos require a read token with access to this model.
try:
    HF_TOKEN = userdata.get("HF_TOKEN")
except Exception:
    HF_TOKEN = None
HF_TOKEN = HF_TOKEN.strip() if HF_TOKEN and HF_TOKEN.strip() else None
if HF_TOKEN is None:
    print("No HF_TOKEN secret found. Public repos can load without it; for a private/gated repo add a read token as the Colab Secret HF_TOKEN.")

tokenizer_files = {"tokenizer.json", "tokenizer_config.json", "special_tokens_map.json", "vocab.json", "tokenizer.model"}
tokenizer_source = ADAPTER_DIR if tokenizer_files.intersection(adapter_files) else BASE_MODEL_NAME
try:
    tokenizer = AutoTokenizer.from_pretrained(tokenizer_source, token=HF_TOKEN, trust_remote_code=True)
except OSError as exc:
    detail = str(exc).lower()
    if any(marker in detail for marker in ("401", "unauthorized", "repositorynotfound", "gatedrepo", "access denied")):
        raise RuntimeError(f"Hugging Face denied access to {tokenizer_source}. Verify the exact repo ID; if it is private/gated, accept its access terms and add a read token with access as Colab Secret HF_TOKEN.") from exc
    raise RuntimeError(f"Could not load tokenizer from {tokenizer_source}; verify the repo/path and tokenizer files.") from exc
except Exception as exc:
    raise RuntimeError(f"Could not load tokenizer from {tokenizer_source} ({type(exc).__name__}). Check model access and adapter tokenizer files.") from exc
if tokenizer.pad_token_id is None:
    if tokenizer.eos_token_id is None:
        raise RuntimeError("Tokenizer has neither a pad token nor an EOS token; cannot configure padding safely.")
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "left"
print(f"Tokenizer loaded from {tokenizer_source}; pad_token_id={tokenizer.pad_token_id}")

In [ ]:
import gc
from transformers import AutoModelForCausalLM

# Release allocations if this notebook cell is being rerun in the same Colab runtime.
for variable_name in ("model", "base_model"):
    if variable_name in globals():
        del globals()[variable_name]
gc.collect()
torch.cuda.empty_cache()

try:
    base_model = AutoModelForCausalLM.from_pretrained(
        BASE_MODEL_NAME,
        quantization_config=quantization_config,
        torch_dtype=torch.float16,
        device_map="auto",
        low_cpu_mem_usage=True,
        token=HF_TOKEN,
        # Let Transformers select a supported attention backend for this runtime.
    )
except torch.cuda.OutOfMemoryError as exc:
    torch.cuda.empty_cache()
    raise RuntimeError("Not enough free GPU memory to load the 4-bit base model. Restart the Colab runtime, select a T4 GPU, and rerun from the start.") from exc
except OSError as exc:
    detail = str(exc).lower()
    if any(marker in detail for marker in ("401", "unauthorized", "repositorynotfound", "gatedrepo", "access denied")):
        raise RuntimeError(f"Hugging Face denied access to {BASE_MODEL_NAME}. Verify the exact repo ID; if it is private/gated, accept its access terms and add a read token with access as Colab Secret HF_TOKEN.") from exc
    raise RuntimeError(f"Could not load base model {BASE_MODEL_NAME}; check the repo ID, model files, and installed transformers/bitsandbytes versions.") from exc
except Exception as exc:
    raise RuntimeError(f"Base Qwen3 model load failed ({type(exc).__name__}). Check GPU availability, model files, and installed transformers/bitsandbytes versions.") from exc

In [ ]:
from peft import PeftModel
try:
    model = PeftModel.from_pretrained(base_model, ADAPTER_DIR, is_trainable=False)
    model.eval()
    model.config.use_cache = True
    model.config.pad_token_id = tokenizer.pad_token_id
    model.generation_config.pad_token_id = tokenizer.pad_token_id
except Exception as exc:
    raise RuntimeError("Could not attach the saved LoRA adapter. Confirm it matches the configured base model and remains intact in Drive.") from exc

In [ ]:
if model.config.pad_token_id != tokenizer.pad_token_id:
    raise RuntimeError("Model/tokenizer padding token IDs do not match.")
if not getattr(model, "peft_config", None):
    raise RuntimeError("PEFT adapter is not attached to the loaded model.")
_MODEL_READY = True
print(f"Ready: {BASE_MODEL_NAME} + LoRA adapter; eval={not model.training}; device={model.device}; pad_token_id={tokenizer.pad_token_id}")

In [ ]:
SYSTEM_PROMPT = (
    "You are CampusGuide AI, the SmartCampus university orientation assistant. "
    "Reply in the language the student uses; support French, Tunisian Arabic (Derja), and Arabizi. "
    "Be friendly, natural, clear, and helpful about university orientation. "
    "Never invent official university information, admission requirements, programmes, tuition fees, or registration dates. "
    "If reliable information is not available in the conversation, say clearly that you are unsure and suggest checking official university channels. "
    "Never claim to have completed registration or any administrative operation. "
    "Do not request passwords, payment details, identity-document numbers, or other sensitive personal information."
)

In [ ]:
import threading
INFERENCE_LOCK = threading.Lock()

def generate_answer(message, history=None, language=None):
    if not isinstance(message, str) or not message.strip():
        raise ValueError("message must be a non-empty string")
    user_text = message.strip()
    system_text = SYSTEM_PROMPT
    if language and str(language).strip():
        system_text += f" The requested language identifier is {str(language).strip()!r}; follow it unless the student clearly writes in another language."
    turns = []
    history_items = (history or [])[-(MAX_HISTORY_TURNS * 2):] if MAX_HISTORY_TURNS > 0 else []
    for item in history_items:
        role = item.get("role") if isinstance(item, dict) else getattr(item, "role", None)
        content = item.get("content") if isinstance(item, dict) else getattr(item, "content", None)
        if role in ("user", "assistant") and isinstance(content, str) and content.strip():
            turns.append({"role": role, "content": content.strip()})
    messages = [{"role": "system", "content": system_text}, *turns, {"role": "user", "content": user_text}]
    context_limit = min(int(getattr(model.config, "max_position_embeddings", MAX_CONTEXT_TOKENS)), MAX_CONTEXT_TOKENS)
    input_limit = context_limit - MAX_NEW_TOKENS
    if input_limit < 128:
        raise RuntimeError("Generation configuration leaves too little room for the prompt.")

    def encode_chat(items):
        try:
            encoded = tokenizer.apply_chat_template(
                items, tokenize=True, add_generation_prompt=True, return_tensors="pt", enable_thinking=False
            )
        except TypeError:
            encoded = tokenizer.apply_chat_template(items, tokenize=True, add_generation_prompt=True, return_tensors="pt")
        return encoded["input_ids"] if isinstance(encoded, dict) else encoded

    input_ids = encode_chat(messages)
    while input_ids.shape[-1] > input_limit and turns:
        turns.pop(0)
        messages = [{"role": "system", "content": system_text}, *turns, {"role": "user", "content": user_text}]
        input_ids = encode_chat(messages)
    while input_ids.shape[-1] > input_limit and len(user_text) > 256:
        user_text = user_text[len(user_text) // 4:]
        messages[-1]["content"] = user_text
        input_ids = encode_chat(messages)
    if input_ids.shape[-1] > input_limit:
        raise ValueError("Message is too long for the model context window; shorten it and try again.")

    input_ids = input_ids.to(model.device)
    attention_mask = torch.ones_like(input_ids)
    options = {
        "max_new_tokens": MAX_NEW_TOKENS,
        "repetition_penalty": REPETITION_PENALTY,
        "pad_token_id": tokenizer.pad_token_id,
        "eos_token_id": tokenizer.eos_token_id,
        "use_cache": True,
    }
    if TEMPERATURE > 0:
        options.update(do_sample=True, temperature=TEMPERATURE, top_p=TOP_P)
    else:
        options["do_sample"] = False
    try:
        with INFERENCE_LOCK, torch.inference_mode():
            output = model.generate(input_ids=input_ids, attention_mask=attention_mask, **options)
        new_tokens = output[0, input_ids.shape[-1]:]
        return tokenizer.decode(new_tokens, skip_special_tokens=True).strip()
    finally:
        del input_ids, attention_mask
        if "output" in locals():
            del output
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

In [ ]:
print(generate_answer("Quels services d'orientation propose SmartCampus AI ?", language="fr"))

In [ ]:
print(generate_answer("Chnowa najm naamel bech naaref anehou parcours ynasebni?", language="derja"))

In [ ]:
print(generate_answer("Chnowa les etapes bech na3mel orientation universitaire?", language="arabizi"))

In [ ]:
from typing import Literal, Optional
from pydantic import BaseModel, ConfigDict, Field, field_validator

class HistoryMessage(BaseModel):
    model_config = ConfigDict(extra="forbid")
    role: Literal["user", "assistant"]
    content: str = Field(min_length=1, max_length=10000)

class ChatRequest(BaseModel):
    model_config = ConfigDict(extra="forbid")
    message: str = Field(min_length=1, max_length=10000)
    history: list[HistoryMessage] = Field(default_factory=list, max_length=40)
    language: Optional[str] = Field(default=None, max_length=32)

    @field_validator("message")
    @classmethod
    def message_not_blank(cls, value):
        if not value.strip():
            raise ValueError("message cannot be blank")
        return value.strip()

class ChatResponse(BaseModel):
    success: bool
    answer: str

In [ ]:
from fastapi import Depends, FastAPI, Header, HTTPException
import hmac
import logging

app = FastAPI(title="SmartCampus AI Inference API", version=API_VERSION)
api_logger = logging.getLogger("smartcampus.inference")
_MODEL_READY = True

In [ ]:
def api_root():
    return {
        "application": "SmartCampus AI Inference API",
        "version": API_VERSION,
        "model": BASE_MODEL_NAME,
        "status": "ready" if _MODEL_READY else "loading",
    }

def health():
    status = "healthy" if _MODEL_READY else "unavailable"
    return {"status": status, "model_loaded": _MODEL_READY, "model": BASE_MODEL_NAME}

def chat_endpoint(payload: ChatRequest):
    if not _MODEL_READY:
        raise HTTPException(status_code=503, detail="Model is not ready.")
    try:
        answer = generate_answer(
            payload.message,
            history=[entry.model_dump() for entry in payload.history],
            language=payload.language,
        )
        if not answer:
            raise RuntimeError("Model generated an empty answer.")
        return ChatResponse(success=True, answer=answer)
    except ValueError as exc:
        raise HTTPException(status_code=422, detail=str(exc)) from exc
    except HTTPException:
        raise
    except Exception as exc:
        api_logger.exception("Inference request failed")
        raise HTTPException(status_code=500, detail="Inference failed. Check server logs.") from exc

app.add_api_route("/", api_root, methods=["GET"])
app.add_api_route(HEALTH_ENDPOINT, health, methods=["GET"])

In [ ]:
from google.colab import userdata
REQUIRE_API_KEY = False
INFERENCE_API_KEY = None
try:
    INFERENCE_API_KEY = userdata.get("INFERENCE_API_KEY")
except Exception:
    INFERENCE_API_KEY = None
if INFERENCE_API_KEY:
    REQUIRE_API_KEY = True
    print("Optional INFERENCE_API_KEY secret found; /chat will require X-API-Key.")
else:
    print("No INFERENCE_API_KEY secret found. Public /chat will have no API authentication; add the secret before exposing sensitive workloads.")

def require_api_key(x_api_key: Optional[str] = Header(default=None, alias="X-API-Key")):
    if REQUIRE_API_KEY and (not x_api_key or not hmac.compare_digest(x_api_key, INFERENCE_API_KEY)):
        raise HTTPException(status_code=401, detail="Invalid or missing API key.")

app.add_api_route(
    CHAT_ENDPOINT,
    chat_endpoint,
    methods=["POST"],
    response_model=ChatResponse,
    dependencies=[Depends(require_api_key)],
)

In [ ]:
from pyngrok import ngrok
try:
    NGROK_AUTHTOKEN = userdata.get("NGROK_AUTHTOKEN")
except Exception as exc:
    raise RuntimeError("Add NGROK_AUTHTOKEN in Colab Secrets (key icon), grant notebook access, then rerun this cell.") from exc
if not NGROK_AUTHTOKEN or not NGROK_AUTHTOKEN.strip():
    raise RuntimeError("NGROK_AUTHTOKEN is missing or empty. Add a valid ngrok authtoken in Colab Secrets.")
ngrok.set_auth_token(NGROK_AUTHTOKEN.strip())
del NGROK_AUTHTOKEN

In [ ]:
import requests
import threading
import time
import uvicorn

server_config = uvicorn.Config(app, host=SERVER_HOST, port=SERVER_PORT, log_level="info")
server = uvicorn.Server(server_config)
server_thread = threading.Thread(target=server.run, name="smartcampus-inference-api", daemon=True)
server_thread.start()
local_base_url = f"http://127.0.0.1:{SERVER_PORT}"
for attempt in range(60):
    if not server_thread.is_alive():
        raise RuntimeError("Uvicorn stopped during startup. Review the server log output above.")
    try:
        response = requests.get(local_base_url + HEALTH_ENDPOINT, timeout=2)
        if response.ok and response.json().get("model_loaded"):
            print(f"Uvicorn is ready on {SERVER_HOST}:{SERVER_PORT}")
            break
    except requests.RequestException:
        pass
    time.sleep(1)
else:
    server.should_exit = True
    raise RuntimeError("Inference server did not become ready within 60 seconds. Check port availability and server logs.")

In [ ]:
try:
    tunnel = ngrok.connect(addr=SERVER_PORT, proto="http", bind_tls=True)
    public_base_url = tunnel.public_url
    if not public_base_url.startswith("https://"):
        raise RuntimeError("ngrok did not provide an HTTPS tunnel.")
    print("HTTPS tunnel established.")
except Exception as exc:
    raise RuntimeError(f"Could not create ngrok tunnel to configured port {SERVER_PORT}. Check your ngrok account/token and port.") from exc

In [ ]:
CHAT_URL = public_base_url.rstrip("/") + CHAT_ENDPOINT
HEALTH_URL = public_base_url.rstrip("/") + HEALTH_ENDPOINT
print(f"Public base URL: {public_base_url}")
print(f"Chat endpoint:   {CHAT_URL}")
print(f"Health endpoint: {HEALTH_URL}")
print("Configure the existing backend with this base URL. Do not have the browser/frontend call the tunnel directly.")

In [ ]:
headers = {"X-API-Key": INFERENCE_API_KEY} if REQUIRE_API_KEY else {}
health_response = requests.get(HEALTH_URL, timeout=20)
health_response.raise_for_status()
print("Health:", health_response.json())
chat_response = requests.post(
    CHAT_URL,
    headers=headers,
    json={
        "message": "Chnowa les parcours disponibles fel IIT?",
        "history": [],
        "language": "derja",
    },
    timeout=240,
)
chat_response.raise_for_status()
print("Chat:", chat_response.json())

In [ ]:
def stop_inference_server():
    global tunnel
    if "server" in globals():
        server.should_exit = True
        server_thread.join(timeout=15)
        if server_thread.is_alive():
            raise RuntimeError("Uvicorn did not stop within 15 seconds.")
    if "tunnel" in globals() and tunnel is not None:
        ngrok.disconnect(tunnel.public_url)
        tunnel = None
    ngrok.kill()
    print("Inference server and ngrok tunnel stopped.")

# Run stop_inference_server() when finished. Keep the Colab runtime active while the backend needs this API.